In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")          # let the notebook see the project root, where src/ lives

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.features import clean, add_features

df = pd.read_csv("../data/raw/application_train.csv")
df = clean(df)
df = add_features(df)

X = df.drop(columns=["TARGET", "SK_ID_CURR"])
y = df["TARGET"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)

(246008, 127) (61503, 127)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
from src.features import CategoryEncoder

def make_model(**xgb_params):
    params = dict(n_estimators=300, max_depth=3, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8, enable_categorical=True,
                  eval_metric="auc", n_jobs=-1, random_state=42)
    params.update(xgb_params)
    return Pipeline([
        ("categories", CategoryEncoder()),
        ("xgb", XGBClassifier(**params)),
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(make_model(), X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)
print("CV AUC:", scores.round(4), "mean", scores.mean().round(4))

CV AUC: [0.759  0.7584 0.7612 0.7598 0.7608] mean 0.7598


In [3]:
import time

candidates = [
    dict(max_depth=3, learning_rate=0.05, n_estimators=300),   # current
    dict(max_depth=4, learning_rate=0.05, n_estimators=300),
    dict(max_depth=5, learning_rate=0.05, n_estimators=300),
    dict(max_depth=4, learning_rate=0.03, n_estimators=600),
]

results = []
for params in candidates:
    start = time.time()
    s = cross_val_score(make_model(**params), X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)
    results.append({**params, "cv_auc": s.mean(), "cv_std": s.std(), "seconds": time.time() - start})
    print(params, "→", round(s.mean(), 4))

pd.DataFrame(results).sort_values("cv_auc", ascending=False)

{'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 300} → 0.7598
{'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 300} → 0.7624
{'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 300} → 0.7624
{'max_depth': 4, 'learning_rate': 0.03, 'n_estimators': 600} → 0.7635


,max_depth,learning_rate,n_estimators,cv_auc,cv_std,seconds
3,4,0.03,600,0.763466,0.001101,34.833138
2,5,0.05,300,0.762364,0.001108,23.534340
1,4,0.05,300,0.762359,0.001129,21.948807
0,3,0.05,300,0.759842,0.001039,14.053077


In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

best_params = dict(max_depth=4, learning_rate=0.03, n_estimators=600)

final_model = make_model(**best_params)
final_model.fit(X_train, y_train)

p_test = final_model.predict_proba(X_test)[:, 1]
base = np.full(len(y_test), y_train.mean())

print("TEST ROC-AUC :", round(roc_auc_score(y_test, p_test), 4))
print("TEST PR-AUC  :", round(average_precision_score(y_test, p_test), 4), "  random ≈", round(y_test.mean(), 4))
print("TEST log loss:", round(log_loss(y_test, p_test), 4), "  baseline:", round(log_loss(y_test, base), 4))

TEST ROC-AUC : 0.7679
TEST PR-AUC  : 0.2579   random ≈ 0.0807
TEST log loss: 0.243   baseline: 0.2805


In [5]:
cal = pd.DataFrame({"predicted": p_test, "actual": y_test.values})
cal["decile"] = pd.qcut(cal["predicted"], 10, labels=False) + 1
cal.groupby("decile").agg(avg_predicted=("predicted", "mean"),
                          actual_rate=("actual", "mean")).round(3)

,avg_predicted,actual_rate
decile,,
1,0.013,0.011
2,0.021,0.018
3,0.028,0.026
4,0.037,0.036
5,0.046,0.043
6,0.059,0.063
7,0.076,0.074
8,0.102,0.102
9,0.146,0.151


In [5]:
import json, joblib, sklearn, xgboost
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

joblib.dump(final_model, models_dir / "credit_model.joblib")

metadata = {
    "model": "CategoryEncoder + XGBClassifier",
    "params": best_params,
    "features": list(X_train.columns),
    "train_rows": len(X_train),
    "test_metrics": {
        "roc_auc": round(roc_auc_score(y_test, p_test), 4),
        "pr_auc": round(average_precision_score(y_test, p_test), 4),
        "log_loss": round(log_loss(y_test, p_test), 4),
    },
    "versions": {"sklearn": sklearn.__version__, "xgboost": xgboost.__version__, "pandas": pd.__version__},
}
(models_dir / "credit_model_meta.json").write_text(json.dumps(metadata, indent=2))
print("saved:", [p.name for p in models_dir.iterdir()])

saved: ['credit_model.joblib', 'credit_model_meta.json']


In [6]:
loaded = joblib.load(models_dir / "credit_model.joblib")

# sanity check: the loaded model gives identical predictions
print("identical:", np.allclose(loaded.predict_proba(X_test)[:, 1], p_test))

# a "new applicant": one raw row, exactly as it would arrive (before any cleaning)
raw = pd.read_csv("../data/raw/application_train.csv")
applicant = raw.loc[[X_test.index[8]]].drop(columns=["TARGET"])
applicant_features = add_features(clean(applicant)).drop(columns=["SK_ID_CURR"])

pd_estimate = loaded.predict_proba(applicant_features)[0, 1]
print(f"applicant {applicant['SK_ID_CURR'].iloc[0]}: P(default) = {pd_estimate:.3f}")
print("approve" if pd_estimate < 0.182 else "reject", "(break-even cutoff 0.182)")

identical: True
applicant 218614: P(default) = 0.360
reject (break-even cutoff 0.182)


In [7]:
from src.predict import predict

new_applicants = raw.loc[X_test.index[:5]].drop(columns=["TARGET"])
predict(new_applicants)

,SK_ID_CURR,pd_default,decision
0,396899,0.068982,approve
1,322041,0.046804,approve
2,220127,0.204771,reject
3,251531,0.068620,approve
4,345558,0.077970,approve


In [8]:
np.allclose(predict(new_applicants)["pd_default"], p_test[:5])

True